# Create project data


# Create Project Data

## Purpose
This notebook loads the raw NBA CSV files, validates their structure, registers them in DuckDB, performs basic SQL-based quality checks, and writes cleaned versions of the project tables for downstream analysis.

## Data Sources
- game.csv
- line_score.csv
- team_history.csv
- other_stats.csv

## Data Loading and Validation

## DuckDB Registration and SQL Checks

## Data Cleaning

## Export Cleaned Tables

## Summary

In [3]:
import logging
from pathlib import Path

import duckdb
import pandas as pd


# ============================================================
# Logging setup
# ============================================================

Path("../logs").mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("../logs/pipeline.log"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)


# ============================================================
# File paths
# ============================================================

RAW_DATA_DIR = Path("../data")
CLEAN_DATA_DIR = Path("../data/cleaned")

GAME_PATH = RAW_DATA_DIR / "game.csv"
LINE_SCORE_PATH = RAW_DATA_DIR / "line_score.csv"
TEAM_HISTORY_PATH = RAW_DATA_DIR / "team_history.csv"
OTHER_STATS_PATH = RAW_DATA_DIR / "other_stats.csv"

CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Helper functions
# ============================================================

def load_csv_file(file_path: Path) -> pd.DataFrame:
    """
    Load a CSV file into a pandas DataFrame.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file: {file_path}")

    logger.info("Loading file: %s", file_path)
    return pd.read_csv(file_path)


def validate_columns(df: pd.DataFrame, required_cols: list[str], table_name: str) -> None:
    """
    Validate that a DataFrame contains required columns.
    """
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")
    logger.info("%s passed column validation", table_name)


def register_tables(
    con: duckdb.DuckDBPyConnection,
    game: pd.DataFrame,
    line_score: pd.DataFrame,
    team_history: pd.DataFrame,
    other_stats: pd.DataFrame
) -> None:
    """
    Register pandas DataFrames as DuckDB tables.
    """
    con.register("game", game)
    con.register("line_score", line_score)
    con.register("team_history", team_history)
    con.register("other_stats", other_stats)
    logger.info("All tables registered in DuckDB")


def basic_clean_game(game: pd.DataFrame) -> pd.DataFrame:
    """
    Apply light cleaning to the game table.
    """
    game_clean = game.copy()

    # Convert game_date to datetime
    game_clean["game_date"] = pd.to_datetime(game_clean["game_date"], errors="coerce")

    # Drop rows missing core identifiers or outcome fields
    game_clean = game_clean.dropna(subset=["game_id", "game_date", "pts_home", "pts_away"])

    # Remove duplicate game rows if present
    game_clean = game_clean.drop_duplicates(subset=["game_id"])

    logger.info("Cleaned game table shape: %s", game_clean.shape)
    return game_clean


def basic_clean_line_score(line_score: pd.DataFrame) -> pd.DataFrame:
    """
    Apply light cleaning to the line_score table.
    """
    line_score_clean = line_score.copy()
    line_score_clean["game_date_est"] = pd.to_datetime(line_score_clean["game_date_est"], errors="coerce")
    line_score_clean = line_score_clean.dropna(subset=["game_id"])
    line_score_clean = line_score_clean.drop_duplicates(subset=["game_id"])
    logger.info("Cleaned line_score table shape: %s", line_score_clean.shape)
    return line_score_clean


def basic_clean_team_history(team_history: pd.DataFrame) -> pd.DataFrame:
    """
    Apply light cleaning to the team_history table.
    """
    team_history_clean = team_history.copy()
    team_history_clean = team_history_clean.dropna(subset=["team_id"])
    logger.info("Cleaned team_history table shape: %s", team_history_clean.shape)
    return team_history_clean


def basic_clean_other_stats(other_stats: pd.DataFrame) -> pd.DataFrame:
    """
    Apply light cleaning to the other_stats table.
    """
    other_stats_clean = other_stats.copy()
    other_stats_clean = other_stats_clean.dropna(subset=["game_id"])
    other_stats_clean = other_stats_clean.drop_duplicates(subset=["game_id"])
    logger.info("Cleaned other_stats table shape: %s", other_stats_clean.shape)
    return other_stats_clean


# ============================================================
# Main workflow
# ============================================================

try:
    logger.info("Starting create_project_data workflow")

    # Load raw CSV files
    game = load_csv_file(GAME_PATH)
    line_score = load_csv_file(LINE_SCORE_PATH)
    team_history = load_csv_file(TEAM_HISTORY_PATH)
    other_stats = load_csv_file(OTHER_STATS_PATH)

    # Validate required columns
    validate_columns(
        game,
        ["game_id", "game_date", "team_id_home", "team_id_away", "pts_home", "pts_away"],
        "game"
    )
    validate_columns(
        line_score,
        ["game_id", "team_id_home", "team_id_away", "pts_home", "pts_away"],
        "line_score"
    )
    validate_columns(
        team_history,
        ["team_id", "city", "nickname", "year_founded", "year_active_till"],
        "team_history"
    )
    validate_columns(
        other_stats,
        ["game_id", "team_id_home", "team_id_away"],
        "other_stats"
    )

    # Show raw table sizes
    print("Raw table shapes:")
    print("game:", game.shape)
    print("line_score:", line_score.shape)
    print("team_history:", team_history.shape)
    print("other_stats:", other_stats.shape)

    # DuckDB connection
    con = duckdb.connect()
    register_tables(con, game, line_score, team_history, other_stats)

    # ========================================================
    # SQL checks
    # ========================================================

    logger.info("Running SQL validation queries")

    sql_counts = con.execute("""
        SELECT 'game' AS table_name, COUNT(*) AS row_count FROM game
        UNION ALL
        SELECT 'line_score' AS table_name, COUNT(*) AS row_count FROM line_score
        UNION ALL
        SELECT 'team_history' AS table_name, COUNT(*) AS row_count FROM team_history
        UNION ALL
        SELECT 'other_stats' AS table_name, COUNT(*) AS row_count FROM other_stats
    """).df()

    print("\nRow counts by table:")
    display(sql_counts)

    sql_season_summary = con.execute("""
        SELECT
            season_type,
            COUNT(*) AS game_count,
            AVG(pts_home) AS avg_home_points,
            AVG(pts_away) AS avg_away_points
        FROM game
        GROUP BY season_type
        ORDER BY game_count DESC
    """).df()

    print("\nSeason type summary:")
    display(sql_season_summary)

    sql_join_check = con.execute("""
        SELECT
            COUNT(*) AS matched_games
        FROM game g
        INNER JOIN line_score l
            ON g.game_id = l.game_id
    """).df()

    print("\nJoin check between game and line_score:")
    display(sql_join_check)

    # ========================================================
    # Cleaning
    # ========================================================

    game_clean = basic_clean_game(game)
    line_score_clean = basic_clean_line_score(line_score)
    team_history_clean = basic_clean_team_history(team_history)
    other_stats_clean = basic_clean_other_stats(other_stats)

    print("\nCleaned table shapes:")
    print("game_clean:", game_clean.shape)
    print("line_score_clean:", line_score_clean.shape)
    print("team_history_clean:", team_history_clean.shape)
    print("other_stats_clean:", other_stats_clean.shape)

    # ========================================================
    # Export cleaned files
    # ========================================================

    game_clean.to_csv(CLEAN_DATA_DIR / "game_clean.csv", index=False)
    line_score_clean.to_csv(CLEAN_DATA_DIR / "line_score_clean.csv", index=False)
    team_history_clean.to_csv(CLEAN_DATA_DIR / "team_history_clean.csv", index=False)
    other_stats_clean.to_csv(CLEAN_DATA_DIR / "other_stats_clean.csv", index=False)

    logger.info("Exported cleaned CSV files to %s", CLEAN_DATA_DIR)

    print("\nCleaned files written to:")
    print(CLEAN_DATA_DIR.resolve())

    # ========================================================
    # Final preview
    # ========================================================

    print("\nPreview of cleaned game table:")
    display(game_clean.head())

    logger.info("create_project_data workflow completed successfully")

except FileNotFoundError as e:
    logger.error("File path error: %s", e)
    print(f"File path error: {e}")

except ValueError as e:
    logger.error("Validation error: %s", e)
    print(f"Validation error: {e}")

except pd.errors.ParserError as e:
    logger.error("CSV parsing error: %s", e)
    print(f"CSV parsing error: {e}")

except duckdb.Error as e:
    logger.error("DuckDB error: %s", e)
    print(f"DuckDB error: {e}")

except Exception as e:
    logger.exception("Unexpected error during create_project_data workflow")
    print(f"Unexpected error: {e}")

2026-03-30 14:43:07,505 - INFO - Starting create_project_data workflow
2026-03-30 14:43:07,510 - INFO - Loading file: ../data/game.csv
2026-03-30 14:43:08,001 - INFO - Loading file: ../data/line_score.csv
2026-03-30 14:43:08,204 - INFO - Loading file: ../data/team_history.csv
2026-03-30 14:43:08,210 - INFO - Loading file: ../data/other_stats.csv
2026-03-30 14:43:08,272 - INFO - game passed column validation
2026-03-30 14:43:08,273 - INFO - line_score passed column validation
2026-03-30 14:43:08,273 - INFO - team_history passed column validation
2026-03-30 14:43:08,274 - INFO - other_stats passed column validation


Raw table shapes:
game: (65698, 55)
line_score: (58053, 43)
team_history: (52, 5)
other_stats: (28271, 26)


2026-03-30 14:43:08,503 - INFO - All tables registered in DuckDB
2026-03-30 14:43:08,504 - INFO - Running SQL validation queries



Row counts by table:


,table_name,row_count
0,game,65698
1,line_score,58053
2,team_history,52
3,other_stats,28271


...


Cleaned table shapes:
game_clean: (65642, 55)
line_score_clean: (58013, 43)
team_history_clean: (52, 5)
other_stats_clean: (28261, 26)


2026-03-30 14:43:11,173 - INFO - Exported cleaned CSV files to ../data/cleaned



Cleaned files written to:
/Users/ben/Project-1-Relational-Model_bengarozzo/data/cleaned

Preview of cleaned game table:


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01,HUS vs. NYK,L,0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season


2026-03-30 14:43:11,188 - INFO - create_project_data workflow completed successfully
